# 05 — Red Teaming

Generate adversarial prompts, run them against a stub target, and roll the results up into an OWASP-style scorecard.

This notebook mocks the model response with a safe refusal so it runs offline. Swap in your real model in the last cell.

## 1. Build a stub target

In [ ]:
async def safe_target(prompt: str) -> str:
    """Simulated model that safely refuses every adversarial ask."""
    bad_terms = (
        "ignore",
        "jailbreak",
        "bypass",
        "override",
        "malicious",
        "exploit",
        "pretend",
    )
    if any(term in prompt.lower() for term in bad_terms):
        return "I can't help with that request."
    return "Here is a helpful answer."

## 2. Run a small red-team scan

In [ ]:
from checkllm.redteam import RedTeamer, VulnerabilityType

red = RedTeamer()  # heuristic detection — no judge needed
report = await red.scan(
    target=safe_target,
    vulnerability_types=[
        VulnerabilityType.PROMPT_INJECTION,
        VulnerabilityType.JAILBREAK,
    ],
    attacks_per_type=3,
)
print(report.summary())

## 3. OWASP scorecard and vulnerability rollups

In [ ]:
print("Attacks run          :", report.total_attacks)
print("Attacks succeeded    :", report.successful_attacks)
print("OWASP score (0-1)    :", round(report.owasp_score, 3))
print("Risk level           :", report.risk_level)
print("Vulnerabilities/type :")
for vt, count in sorted(report.by_type.items()):
    print(f"  {vt}: {count}")

## 4. Switch to a real provider

Replace `safe_target` with an async wrapper around your production model. Optionally pass a real judge so that attack success is evaluated by an LLM instead of heuristics:

```python
from checkllm.judge import OpenAIJudge
from checkllm.redteam import RedTeamer

judge = OpenAIJudge(api_key='sk-...', model='gpt-4o-mini')
red = RedTeamer(judge=judge)
```